**Checking Yearly Draft Pick Players**

Change the year for each season

In [ ]:
!pip install nba_api
from nba_api.stats.endpoints import drafthistory
import pandas as pd

# Fetch draft history with the 'season_year' parameter set to '2025'
draft_2025 = drafthistory.DraftHistory(season_year_nullable='2025')

# Convert the data into a pandas DataFrame
df_2025_draft = draft_2025.get_data_frames()[0]

# Print the DataFrame to see the results
print(df_2025_draft)


    PERSON_ID               PLAYER_NAME SEASON  ROUND_NUMBER  ROUND_PICK  OVERALL_PICK DRAFT_TYPE     TEAM_ID      TEAM_CITY      TEAM_NAME TEAM_ABBREVIATION                    ORGANIZATION   ORGANIZATION_TYPE  PLAYER_PROFILE_FLAG
0     1642843              Cooper Flagg   2025             1           1             1      Draft  1610612742         Dallas      Mavericks               DAL                            Duke  College/University                    1
1     1642844              Dylan Harper   2025             1           2             2      Draft  1610612759    San Antonio          Spurs               SAS                         Rutgers  College/University                    1
2     1642845              VJ Edgecombe   2025             1           3             3      Draft  1610612755   Philadelphia          76ers               PHI                          Baylor  College/University                    1
3     1642851              Kon Knueppel   2025             1           4    

**Automated Player Trade Detection for Daily Games**



In [ ]:
import pandas as pd
from datetime import date
import time
from nba_api.stats.endpoints import leaguedashplayerbiostats, leaguegamefinder
from nba_api.stats.library.parameters import Season

def get_todays_games():
    """Fetches the game schedule for today."""
    today = date.today().strftime('%m/%d/%Y')
    gamefinder = leaguegamefinder.LeagueGameFinder(date_from_nullable=today, date_to_nullable=today)
    games = gamefinder.get_data_frames()[0]
    return games

def get_player_roster():
    """Fetches the current biographical data for all players."""
    season = Season.current_season_year
    all_players_df = leaguedashplayerbiostats.LeagueDashPlayerBioStats(season=season).get_data_frames()[0]
    roster = all_players_df[['PERSON_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION']]
    return roster

def check_trades_for_tonights_games():
    """
    Compares the current roster to a saved baseline to detect trades
    for players involved in tonight's games.
    """
    # Define a filename for your roster baseline
    roster_filename = 'roster_baseline.csv'

    # Get today's games to identify relevant teams
    todays_games = get_todays_games()
    if todays_games.empty:
        print("No games are scheduled for today.")
        return None

    # Get the list of teams playing tonight
    teams_playing_tonight = set(todays_games['TEAM_ABBREVIATION'].tolist())

    # Get the fresh roster from the API
    current_roster = get_player_roster()

    # Try to load the saved baseline roster
    try:
        baseline_roster = pd.read_csv(roster_filename)
    except FileNotFoundError:
        # If no baseline exists, save the current one and exit
        print("No baseline roster found. Creating a new one.")
        current_roster.to_csv(roster_filename, index=False)
        return pd.DataFrame() # Return an empty DataFrame

    # Merge the two rosters to find changes
    comparison_df = pd.merge(
        baseline_roster,
        current_roster,
        on='PERSON_ID',
        suffixes=('_baseline', '_current')
    )

    # Identify players who have switched teams
    traded_players = comparison_df[comparison_df['TEAM_ABBREVIATION_baseline'] != comparison_df['TEAM_ABBREVIATION_current']].copy()

    # Filter for trades involving teams playing tonight
    trades_tonight = traded_players[
        traded_players['TEAM_ABBREVIATION_baseline'].isin(teams_playing_tonight) |
        traded_players['TEAM_ABBREVIATION_current'].isin(teams_playing_tonight)
    ]

    # Update the baseline for the next run
    current_roster.to_csv(roster_filename, index=False)

    return trades_tonight

if __name__ == '__main__':
    print("Checking for trades involving teams playing tonight...")
    trades = check_trades_for_tonights_games()

    if not trades.empty:
        print("\n--- Detected Player Trades for Tonight's Games ---")
        print(trades[['PLAYER_NAME', 'TEAM_ABBREVIATION_baseline', 'TEAM_ABBREVIATION_current']])
    else:
        print("\nNo relevant trades detected since the last check.")


**Simple Odds API Test Script for Props**

In [ ]:
# ==============================================================================
# Simple Odds API Test Script
# ==============================================================================

import requests
import json
import os
from datetime import datetime

# ==============================================================================
# --- CONFIGURATION ---
# ==============================================================================
# THE ONLY THING YOU NEED TO CHANGE: Paste your newest, fresh API key here.
ODDS_API_KEY = [
"9b96859c5e0781bb4c5809215a5ed94b", "f4cfcea7cc6120a97559f69d6049596e", "7b9405e2febc561e554b7169a42cefd1", "146dff42ae19a83208b059d4855095c9", "f8474479351ccc685ccae9f4716c23c5", "39756c093848b6968c899a4e1ded85f9", "9705a1f6b609305047eb8ec9243cc54a",
"4217574ccdd90732483d98f24772f66b",
"66953d96fcb2ec72a47e7f39901258e7", "94304f58fb95fd1d45159b9c8d292724",
"e384411e342cac5c9408804c622703f1",
"dab5c3da80b0415f64d4d94890974cdf",
"9705a1f6b609305047eb8ec9243cc54a",
"3aae94b51cc3dcb8e0623bf2633a695a",
"a4e4d3eef7d360055b96a1351560817f",
"bd6c88c66ff4c7c057aafaffcdd1b8d",
"1bfcac154fb2b490bf55a80a18275db1",
"7b64d450f91079f60211cb33033594b4",
"ec9dc1e1aa0543c1b3fa3d2576fabffd",
"6bbfe433274aef6fee5aa5543ad69055",
"1f5e9cdc190854c605b533917b2c9b82",
"858b7234b5f77661345f205b1c5ec121",
"39756c093848b6968c899a4e1ded85f9",
"21caebe8058c338a7ba76f6c9fb550cb",
"94304f58fb95fd1d45159b9c8d292724",
]
# ==============================================================================
# --- TEST FUNCTIONS ---
# ==============================================================================

def find_first_nba_event_id(api_key):
    """
    Makes a single call to the /events endpoint to find a valid game ID.
    This call should succeed even on a free plan.
    """
    print("--- Step 1: Finding an upcoming NBA game event ID ---")
    url = 'https://api.the-odds-api.com/v4/sports/basketball_nba/events'
    params = {'apiKey': api_key}

    try:
        response = requests.get(url, params=params, timeout=30)
        # Check for any HTTP error (like 401 Unauthorized)
        response.raise_for_status()

        events = response.json()

        if not events:
            print("Result: No upcoming NBA events found in the API response.")
            return None

        # Success! Grab the first event.
        first_event = events[0]
        event_id = first_event['id']
        home_team = first_event['home_team']
        away_team = first_event['away_team']

        print(f"✅ Success! Found a game: {away_team} at {home_team}")
        print(f"   Event ID: {event_id}")
        return event_id

    except requests.exceptions.RequestException as e:
        print(f"❌ CRITICAL FAILURE in Step 1.")
        print(f"   Could not connect to the API or the key is invalid.")
        print(f"   Error: {e}")
        return None

def get_player_props_for_event(event_id, api_key):
    """
    Makes a single call to the /events/{eventId}/odds endpoint.
    This is the crucial test to see if your key has permission for player props.
    """
    if not event_id:
        return

    print(f"\n--- Step 2: Attempting to fetch player props for event ID: {event_id} ---")

    # We will request all possible player prop markets.
    markets = 'player_points,player_rebounds,player_assists,player_points_rebounds_assists,player_points_rebounds,player_points_assists,player_rebounds_assists,player_double_double,player_triple_double'

    url = f'https://api.the-odds-api.com/v4/sports/basketball_nba/events/{event_id}/odds'
    params = {
        'apiKey': api_key,
        'regions': 'us',
        'markets': markets,
        'bookmakers': 'draftkings,fanduel'
    }

    try:
        response = requests.get(url, params=params, timeout=30)

        print(f"   Response Status Code: {response.status_code}")

        # This will raise an error if the status code is 4xx or 5xx
        response.raise_for_status()

        odds_data = response.json()

        print("\n✅✅✅ VICTORY! Successfully fetched player prop data! ✅✅✅")
        print("   This proves your API key has the correct permissions.")
        print("\n--- Sample of Raw Data Received: ---")
        print(json.dumps(odds_data, indent=2))

    except requests.exceptions.RequestException as e:
        print(f"\n❌❌❌ FAILURE in Step 2. ❌❌❌")
        print(f"   The request for player props failed.")
        print(f"   Error: {e}")
        print("\n--- Raw Error Response from Server: ---")
        try:
            # Try to print the server's error message for more clues
            print(e.response.json())
        except:
            print(e.response.text)
        print("\nThis almost certainly confirms the API key's subscription plan does not allow access to this endpoint.")

# ==============================================================================
# --- RUN THE TEST ---
# ==============================================================================
if __name__ == "__main__":
    if ODDS_API_KEY == "PASTE_YOUR_FRESH_API_KEY_HERE" or not ODDS_API_KEY:
        print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
        print("!!! ERROR: Please paste your new API key into the script. !!!")
        print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    else:
        # Run the two-step test
        game_event_id = find_first_nba_event_id(ODDS_API_KEY)
        get_player_props_for_event(game_event_id, ODDS_API_KEY)

--- Step 1: Finding an upcoming NBA game event ID ---
✅ Success! Found a game: Brooklyn Nets at Orlando Magic
   Event ID: b042c1ffa4d1465733e453db2b3beacc

--- Step 2: Attempting to fetch player props for event ID: b042c1ffa4d1465733e453db2b3beacc ---
   Response Status Code: 401

❌❌❌ FAILURE in Step 2. ❌❌❌
   The request for player props failed.
   Error: 401 Client Error: Unauthorized for url: https://api.the-odds-api.com/v4/sports/basketball_nba/events/b042c1ffa4d1465733e453db2b3beacc/odds?apiKey=9b96859c5e0781bb4c5809215a5ed94b&apiKey=f4cfcea7cc6120a97559f69d6049596e&apiKey=7b9405e2febc561e554b7169a42cefd1&apiKey=146dff42ae19a83208b059d4855095c9&apiKey=f8474479351ccc685ccae9f4716c23c5&apiKey=39756c093848b6968c899a4e1ded85f9&apiKey=9705a1f6b609305047eb8ec9243cc54a&apiKey=4217574ccdd90732483d98f24772f66b&apiKey=66953d96fcb2ec72a47e7f39901258e7&apiKey=94304f58fb95fd1d45159b9c8d292724&apiKey=e384411e342cac5c9408804c622703f1&apiKey=dab5c3da80b0415f64d4d94890974cdf&apiKey=9705a1f6b60930

Get game schedule for the current day and EST from odds api

In [ ]:
!pip install pytz
import requests
from datetime import date
from dateutil import parser
import pytz  # Import the pytz library

# IMPORTANT: Replace with your actual API key from The Odds API
API_KEY = "94304f58fb95fd1d45159b9c8d292724"

# Set API parameters
SPORT_KEY = "basketball_nba"
REGIONS = "us"
MARKETS = "h2h"
ODDS_FORMAT = "american"

# Define the target timezone as 'America/New_York' (which covers EST/EDT)
est_timezone = pytz.timezone('America/New_York')

# Get today's date in YYYY-MM-DD format based on the local system time (still useful for filtering)
today_str = date.today().isoformat()

# Construct the API request URL
url = f"https://api.the-odds-api.com/v4/sports/{SPORT_KEY}/odds/?apiKey={API_KEY}&regions={REGIONS}&markets={MARKETS}&oddsFormat={ODDS_FORMAT}"

try:
    response = requests.get(url)
    response.raise_for_status()
    games = response.json()

    print(f"Checking for NBA games for today ({today_str} local date, times in EST):")

    if not games:
        print("API returned no upcoming games. Check your API key and usage limits.")
    else:
        # Filter games that commence on the same *local* calendar day as 'today_str'
        # The parser.isoparse() gives a UTC datetime object which we convert to the local date for comparison
        games_today = [
            game for game in games
            if parser.isoparse(game["commence_time"]).astimezone(est_timezone).date().isoformat() == today_str
        ]

        if not games_today:
            print(f"The API returned upcoming games, but none were scheduled for the local date {today_str}.")
            # Optional: Print all upcoming games to see what was returned
            print("--- Upcoming games found (any date, times in EST) ---")
            for game in games:
                commence_datetime_utc = parser.isoparse(game["commence_time"])
                commence_datetime_est = commence_datetime_utc.astimezone(est_timezone) # Convert to EST
                home_team = game["home_team"]
                away_team = game["away_team"]
                print(f"- {away_team} at {home_team} (Starts: {commence_datetime_est.strftime('%Y-%m-%d %I:%M %p EST')})")
        else:
            print("--- Games found for today (times in EST) ---")
            for game in games_today:
                commence_datetime_utc = parser.isoparse(game["commence_time"])
                commence_datetime_est = commence_datetime_utc.astimezone(est_timezone) # Convert to EST
                home_team = game["home_team"]
                away_team = game["away_team"]
                game_id = game['id']
                print(f"- Game ID: {game_id} | {away_team} at {home_team} (Starts: {commence_datetime_est.strftime('%Y-%m-%d %I:%M %p EST')})")

except requests.exceptions.RequestException as e:
    print(f"An error occurred with the request: {e}")



An error occurred with the request: 401 Client Error: Unauthorized for url: https://api.the-odds-api.com/v4/sports/basketball_nba/odds/?apiKey=94304f58fb95fd1d45159b9c8d292724&regions=us&markets=h2h&oddsFormat=american


Example of how to call each game individually for player props

In [ ]:
# ==============================================================================
# HOW TO GET NBA PLAYER PROPS FOR MULTIPLE GAMES
# ==============================================================================
#
# This script automates the process of fetching specified player props for ALL
# available upcoming NBA games from The Odds API.
#
# The API requires a two-step process that must be repeated for each game:
#
# Step 1: Fetch All Upcoming Games
# The script first makes a single API call to the `/events` endpoint. This returns
# a list of all upcoming NBA games, each with a unique `eventId`.
#
# Step 2: Loop and Fetch Props for Each Game
# The script then iterates through the list of games from Step 1. For EACH game,
# it makes a NEW, SEPARATE API call to the `/events/{eventId}/odds` endpoint.
# This call requests the specific player prop markets for that one game.
#
# Step 3: Combine and Display Results
# The data from all the individual prop requests are collected and combined into a
# single, clean table (a pandas DataFrame) for easy analysis and viewing.
#
# --- How to Use ---
# 1. Set your `API_KEY` in the configuration section below.
# 2. (Optional) Adjust the `MARKETS` list to add or remove props.
# 3. Run this entire cell. The script will find all games and fetch their props.
#
# ==============================================================================

import requests
import pandas as pd
import time
from datetime import datetime

# --- 1. Configuration ---
API_KEY = 'YOUR_API_KEY'
SPORT = 'basketball_nba'
REGIONS = 'us'
BOOKMAKERS = 'fanduel,draftkings'

# Define the player prop markets we want to fetch
MARKETS = [
    'player_points',
    'player_rebounds',
    'player_assists',
    'player_threes',
    'player_points_rebounds_assists',
    'player_points_rebounds',
    'player_points_assists',
    'player_rebounds_assists',
    'player_double_double',
    'player_triple_double'
]
MARKETS_STR = ','.join(MARKETS)

# Map API keys to friendly names for the final table
PROP_MAP = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_threes': 'FG3M',
    'player_points_rebounds_assists': 'PRA',
    'player_points_rebounds': 'P+R',
    'player_points_assists': 'P+A',
    'player_rebounds_assists': 'R+A',
    'player_double_double': 'Double-Double',
    'player_triple_double': 'Triple-Double'
}


def get_upcoming_nba_events(api_key):
    """Fetches all upcoming NBA games to get their event IDs."""
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/events?apiKey={api_key}"
    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching events: {e}")
        return None

def get_player_props_for_event(event_id, api_key):
    """Fetches specified player props for a single game using its event ID."""
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/events/{event_id}/odds"
    params = {
        'apiKey': api_key,
        'regions': REGIONS,
        'bookmakers': BOOKMAKERS,
        'markets': MARKETS_STR,
        'oddsFormat': 'american'
    }
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException:
        # Silently fail for a single event, allows the loop to continue
        return None

# --- 2. Main Script Execution ---

if API_KEY == 'YOUR_API_KEY':
    print("Please replace 'YOUR_API_KEY' with your actual API key and run the cell again.")
else:
    # Step 1: Get all upcoming NBA games
    print("Step 1: Fetching all upcoming NBA games...")
    events = get_upcoming_nba_events(API_KEY)

    if not events:
        print("No upcoming NBA games found or failed to fetch events.")
    else:
        print(f"Found {len(events)} upcoming games. Now fetching props for each...")

        # List to hold all parsed data from every game
        all_odds_data = []

        # Step 2: Loop through each game and get its props
        for i, event in enumerate(events):
            event_id = event['id']
            home_team = event['home_team']
            away_team = event['away_team']
            game_info = f"{away_team} @ {home_team}"

            print(f"  ({i+1}/{len(events)}) Fetching props for: {game_info}")

            # Fetch the props for the current game
            props_data = get_player_props_for_event(event_id, API_KEY)

            if props_data:
                # Parse the response and add to our master list
                for bookmaker in props_data.get('bookmakers', []):
                    for market in bookmaker.get('markets', []):
                        market_name = PROP_MAP.get(market['key'], market['key'])
                        for outcome in market.get('outcomes', []):
                            all_odds_data.append({
                                'Game': game_info,
                                'Bookmaker': bookmaker['title'],
                                'Market': market_name,
                                'Player': outcome['description'],
                                'Bet': f"{outcome['name']} {outcome.get('point', '')}".strip(),
                                'Odds': outcome['price']
                            })

            # Be a good citizen and avoid hitting the rate limit
            time.sleep(1)

        # Step 3: Combine and display the results
        if not all_odds_data:
            print("\nFinished. No player prop odds were found for the specified markets.")
        else:
            print("\nFinished fetching all data. Compiling results into a table...")
            df = pd.DataFrame(all_odds_data)

            # Display the final DataFrame
            display(df)

--- Step 1: Finding today's NBA games ---
(API Usage: 478 requests remaining on this key)

Found 9 games for today (2025-11-14):
  1. Game ID: b042c1ffa4d1465733e453db2b3beacc
     Matchup: Brooklyn Nets at Orlando Magic
     Starts:  07:10 PM EST

  2. Game ID: 63016986bd7402f9433def9ec4be4fe1
     Matchup: Miami Heat at New York Knicks
     Starts:  07:10 PM EST

  3. Game ID: eac240dfaaa6377457daa36ac01b777e
     Matchup: Philadelphia 76ers at Detroit Pistons
     Starts:  07:40 PM EST

  4. Game ID: ab89cda07f21ecec76a5b461174ec380
     Matchup: Charlotte Hornets at Milwaukee Bucks
     Starts:  08:10 PM EST

  5. Game ID: e8dd245278646c0b7835609c0d04c4a5
     Matchup: Portland Trail Blazers at Houston Rockets
     Starts:  08:10 PM EST

  6. Game ID: 5d2bab71b12e005da1a9491834c945dc
     Matchup: Los Angeles Lakers at New Orleans Pelicans
     Starts:  08:10 PM EST

  7. Game ID: 2a534db0689c24c8b50c65bd9e0d2c82
     Matchup: Sacramento Kings at Minnesota Timberwolves
     Starts:

In [ ]:
# ==============================================================================
# UTILITY SCRIPT: Team Abbreviation Mapper
# This script reads your historical odds CSV and generates a complete list of
# all unique team abbreviations found in the 'home' and 'away' columns.
# ==============================================================================

import pandas as pd
from google.colab import drive
import os

def generate_team_map_template():
    """
    Loads the historical odds file, finds all unique team abbreviations,
    and prints a Python dictionary template for easy mapping.
    """
    try:
        print("Mounting Google Drive...")
        drive.mount('/content/drive', force_remount=True)
    except Exception as e:
        print(f"Error mounting Google Drive: {e}")
        return

    # --- CONFIGURATION: Ensure this path is correct ---
    # The script uses the name from our working example.
    # If your file is named nba_2008-2025.csv, change it here.
    file_path = "/content/drive/MyDrive/NBA/Historical Data/nba_2008-2025.csv"

    print(f"\nAttempting to read team names from: {file_path}")

    if not os.path.exists(file_path):
        print(f"CRITICAL ERROR: The file was not found at the specified path.")
        print("Please ensure the file name and location are correct.")
        return

    try:
        df = pd.read_csv(file_path)

        # Check if the required columns exist
        if 'home' not in df.columns or 'away' not in df.columns:
            print("CRITICAL ERROR: The CSV file must contain 'home' and 'away' columns.")
            return

        # Concatenate the 'home' and 'away' columns into a single Series
        home_teams = df['home'].dropna().astype(str)
        away_teams = df['away'].dropna().astype(str)
        all_teams = pd.concat([home_teams, away_teams])

        # Get the unique, lowercase, sorted list of team abbreviations
        unique_teams = sorted(all_teams.str.lower().unique())

        print("\n" + "="*50)
        print("✅ Success! Found the following unique team abbreviations:")
        print(unique_teams)
        print("="*50)

        # --- Generate the Python dictionary template ---
        print("\n\n--- Python Dictionary Template ---")
        print("Copy the text below, complete the mappings, and paste it into Script 1.")
        print("-" * 30)
        print("team_map = {")
        for team in unique_teams:
            # Add a placeholder for the user to fill in
            print(f"    '{team}': '',  # Please fill in the uppercase equivalent (e.g., 'POR')")
        print("}")
        print("-" * 30)

    except Exception as e:
        print(f"An error occurred while processing the file: {e}")

# Run the function
if __name__ == "__main__":
    generate_team_map_template()

Mounting Google Drive...
Mounted at /content/drive

Attempting to read team names from: /content/drive/MyDrive/NBA/Historical Data/nba_2008-2025.csv

✅ Success! Found the following unique team abbreviations:
['atl', 'bkn', 'bos', 'cha', 'chi', 'cle', 'dal', 'den', 'det', 'gs', 'hou', 'ind', 'lac', 'lal', 'mem', 'mia', 'mil', 'min', 'no', 'ny', 'okc', 'orl', 'phi', 'phx', 'por', 'sa', 'sac', 'tor', 'utah', 'wsh']


--- Python Dictionary Template ---
Copy the text below, complete the mappings, and paste it into Script 1.
------------------------------
team_map = {
    'atl': '',  # Please fill in the uppercase equivalent (e.g., 'POR')
    'bkn': '',  # Please fill in the uppercase equivalent (e.g., 'POR')
    'bos': '',  # Please fill in the uppercase equivalent (e.g., 'POR')
    'cha': '',  # Please fill in the uppercase equivalent (e.g., 'POR')
    'chi': '',  # Please fill in the uppercase equivalent (e.g., 'POR')
    'cle': '',  # Please fill in the uppercase equivalent (e.g., 'POR')

DvP from hastagbasketball

In [ ]:
#!/usr/bin/env python3
# dvp_updater.py — PropPulse+ v2025 (HashtagBasketball Edition - HTML Scraper)
#
# Scrapes DvP from:
# https://hashtagbasketball.com/nba-defense-vs-position
#
# Output format:
#   dvp[TEAM_ABBR][POS][STAT] = rank
#
# Supported stats:
#   PTS, REB, AST, FG3M

import os
import json
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

CACHE_FILE = "data/dvp_cache.json"
CACHE_HOURS = 12

HASHTAG_URL = "https://hashtagbasketball.com/nba-defense-vs-position"


# ------------------------------------------------------
# Team Mapping (abbreviations used on HashtagBasketball)
# ------------------------------------------------------
TEAM_MAP = {
    "ATL": "ATL", "BOS": "BOS", "BKN": "BKN", "CHA": "CHA", "CHI": "CHI",
    "CLE": "CLE", "DAL": "DAL", "DEN": "DEN", "DET": "DET", "GSW": "GS",
    "GS": "GSW", "HOU": "HOU", "IND": "IND", "LAC": "LAC", "LAL": "LAL",
    "MEM": "MEM", "MIA": "MIA", "MIL": "MIL", "MIN": "MIN", "NOP": "NO",
    "NO": "NOP", "NYK": "NY", "NY": "NYK", "OKC": "OKC", "ORL": "ORL",
    "PHI": "PHI", "PHX": "PHX", "POR": "POR", "SAC": "SAC", "SAS": "SA",
    "SA": "SAS", "TOR": "TOR", "UTA": "UTA", "WAS": "WAS"
}


# ------------------------------------------------------
# Scrape HashtagBasketball HTML
# ------------------------------------------------------
def _fetch_dvp_from_hashtag():
    """
    Scrapes the HashtagBasketball DvP page.
    The full table (Table 4) has 150 rows: 30 teams × 5 positions
    Each row shows: Position, Team, PTS rank, FG%, FT%, 3PM rank, REB rank, AST rank, STL, BLK, TO
    """
    print("[DVP] 🔄 Fetching DvP from HashtagBasketball HTML...")

    # Initialize nested dict structure
    all_teams = set(TEAM_MAP.values())
    dvp = {abbr: {pos: {} for pos in ["PG", "SG", "SF", "PF", "C"]}
           for abbr in all_teams}

    # Map of our stat names to column names in the table
    stat_columns = {
        "PTS": "Sort: PTS",
        "REB": "Sort: REB",
        "AST": "Sort: AST",
        "FG3M": "Sort: 3PM"
    }

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    try:
        # Fetch the page (we can get all stats from one page)
        print(f"[DVP]   Fetching from {HASHTAG_URL}")
        response = requests.get(HASHTAG_URL, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')

        # Find all tables
        tables = soup.find_all('table')
        print(f"[DVP]   Found {len(tables)} tables")

        # We want the largest table (should be 150 rows for all team-position combos)
        target_table = None
        max_rows = 0

        for table in tables:
            try:
                df = pd.read_html(StringIO(str(table)))[0]
                if len(df) > max_rows:
                    max_rows = len(df)
                    target_table = df
            except:
                continue

        if target_table is None:
            print("[DVP] ❌ Could not find data table")
            return dvp

        df = target_table
        print(f"[DVP]   Processing table with {len(df)} rows")

        # Clean column names
        df.columns = df.columns.astype(str)

        # The table has columns: Position, Team, PTS, FG%, FT%, 3PM, REB, AST, STL, BLK, TO
        pos_col = "Sort: Position"
        team_col = "Sort: Team"

        if pos_col not in df.columns or team_col not in df.columns:
            print(f"[DVP] ❌ Missing required columns. Found: {list(df.columns)}")
            return dvp

        # Process each row
        for idx, row in df.iterrows():
            try:
                position = str(row[pos_col]).strip().upper()
                team_raw = str(row[team_col]).strip().upper()

                # Extract team abbreviation (might have numbers after it)
                # e.g., "NY 3" -> "NY"
                team_abbr = ''.join(c for c in team_raw.split()[0] if c.isalpha())

                # Map to standard abbreviation
                team_abbr = TEAM_MAP.get(team_abbr, team_abbr)

                # Validate position
                if position not in ["PG", "SG", "SF", "PF", "C"]:
                    continue

                if team_abbr not in dvp:
                    continue

                # Extract rank values for each stat
                for stat_key, col_name in stat_columns.items():
                    if col_name in df.columns:
                        try:
                            # The value might be like "15.6 1" where the number after space is the rank
                            # Or it might just be a rank number
                            value_str = str(row[col_name]).strip()

                            # Try to extract rank (typically the last number, or first if it's just rank)
                            parts = value_str.split()

                            # If there are multiple numbers, the last one is usually the rank
                            # If just one number, it could be either value or rank
                            if len(parts) >= 2:
                                # Format like "15.6 1" - last number is rank
                                rank = float(parts[-1])
                            else:
                                # Single number - try to parse it
                                rank = float(parts[0])

                            dvp[team_abbr][position][stat_key] = rank

                        except (ValueError, IndexError):
                            pass

            except Exception as e:
                continue

        # Count successful extractions
        total_entries = sum(
            len(dvp[team][pos])
            for team in dvp
            for pos in dvp[team]
        )

        if total_entries > 0:
            print(f"[DVP] ✅ Successfully loaded {total_entries} DvP rank entries")
        else:
            print("[DVP] ⚠️ No data was extracted from table")

    except Exception as e:
        print(f"[DVP] ❌ Error fetching data: {e}")

    return dvp


# ------------------------------------------------------
# Cache Handling
# ------------------------------------------------------
def _load_cache():
    if not os.path.exists(CACHE_FILE):
        return None

    try:
        age = (time.time() - os.path.getmtime(CACHE_FILE)) / 3600
        if age < CACHE_HOURS:
            with open(CACHE_FILE, "r") as f:
                print(f"[DVP] 🟢 Using cached DvP ({age:.1f}h old)")
                return json.load(f)["dvp"]
    except Exception as e:
        print(f"[DVP] ⚠️ Cache load error: {e}")
        pass

    return None


def _save_cache(dvp):
    os.makedirs("data", exist_ok=True)
    with open(CACHE_FILE, "w") as f:
        json.dump({"timestamp": time.time(), "dvp": dvp}, f, indent=2)
    print("[DVP] 💾 Saved DvP cache.")


# ------------------------------------------------------
# Public Loader
# ------------------------------------------------------
def load_dvp_data(force_refresh=False):
    """
    Load Defense vs Position data.
    Returns dict: dvp[TEAM_ABBR][POS][STAT] = rank
    """
    if not force_refresh:
        cached = _load_cache()
        if cached:
            return cached

    dvp = _fetch_dvp_from_hashtag()

    # Check if we got data
    total_entries = sum(
        len(dvp[team][pos])
        for team in dvp
        for pos in dvp[team]
    )

    if total_entries > 0:
        _save_cache(dvp)
        return dvp

    print("[DVP] ⚠️ No DvP available. Returning empty.")
    return {}


# ------------------------------------------------------
# Test function
# ------------------------------------------------------
if __name__ == "__main__":
    print("Testing DvP scraper...")
    dvp_data = load_dvp_data(force_refresh=True)

    if dvp_data:
        # Count total entries
        total = sum(
            len(dvp_data[team][pos])
            for team in dvp_data
            for pos in dvp_data[team]
        )
        print(f"\n[DVP] Total entries: {total}")

        # Print sample data for a few teams
        print("\n[DVP] Sample data:")
        for sample_team in ["NYK", "GSW", "LAL"][:3]:
            if sample_team in dvp_data:
                print(f"\n{sample_team}:")
                for pos in ["PG", "SG", "SF", "PF", "C"]:
                    if dvp_data[sample_team][pos]:
                        print(f"  {pos}: {dvp_data[sample_team][pos]}")
                break
    else:
        print("\n[DVP] ❌ No data retrieved")

Testing DvP scraper...
[DVP] 🔄 Fetching DvP from HashtagBasketball HTML...
[DVP]   Fetching from https://hashtagbasketball.com/nba-defense-vs-position
[DVP]   Found 4 tables
[DVP]   Processing table with 150 rows
[DVP] ✅ Successfully loaded 580 DvP rank entries
[DVP] 💾 Saved DvP cache.

[DVP] Total entries: 580

[DVP] Sample data:

NYK:
  PG: {'PTS': 1.0, 'REB': 3.0, 'AST': 147.0, 'FG3M': 44.0}
  SG: {'PTS': 130.0, 'REB': 61.0, 'AST': 114.0, 'FG3M': 138.0}
  SF: {'PTS': 64.0, 'REB': 34.0, 'AST': 12.0, 'FG3M': 146.0}
  PF: {'PTS': 126.0, 'REB': 105.0, 'AST': 56.0, 'FG3M': 85.0}
  C: {'PTS': 22.0, 'REB': 131.0, 'AST': 11.0, 'FG3M': 27.0}
